Text Loaders

In [ ]:
#Data Ingestion (Text Loader)
from langchain_community.document_loaders import TextLoader
loader = TextLoader("data.txt")
text_documents = loader.load()
text_documents

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [ ]:
#Data Ingestion (Web Loader)
from langchain_community.document_loaders import WebBaseLoader
import bs4
loader = WebBaseLoader("https://9dtechnologies.com/",
                       bs_kwargs=dict(parse_only=bs4.SoupStrainer(["header", "footer", "p", "h1", "h2", "h3", "li"]))
)
web_documents = loader.load()
web_documents

RAG Pipelines

In [ ]:
#Data Ingestion (PDF Loader)
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("../Nova_Labs_Company_Profile.pdf")
mypdf=loader.load()
mypdf

Text splitter/ Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
document= text_splitter.split_documents(mypdf)
document

Vector Embeddings and Vector Store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

from langchain_community.vectorstores import Chroma
# Convert each chunk in `document` to an embedding and store it in Chroma
db = Chroma.from_documents(document, embeddings)




In [ ]:

query="What is the email of Nova Labs?"
# Test: find the chunks closest in meaning to a question
results = db.similarity_search(query)
results[0].page_content

Advanced RAG using Chain and Retriver

In [ ]:
#koadiing and testiong local LLMs
from langchain_ollama import ChatOllama
# num_gpu=0 runs on CPU: the GPU (Vulkan) crashed on longer RAG prompts
llm = ChatOllama(model="qwen2.5:3b", num_gpu=0)

llm.invoke("Say hello in one line").content


In [ ]:
#Prompt Template
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """You are a helpful assistant. Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.
    Context:{context}
    Question: {question}"""
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = db.as_retriever()

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)
#retriever chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response=rag_chain.invoke("What are the three main products of Nova Labs?")
response
